# Pauli-Based Computation

## 1. Introduction
Any quantum circuit can be approximated to arbitrary precision by Clifford gates ($H$, $S$, $\mathrm{CNOT}$) and $T$ gates

### Fault-tolerant quantum computing

In fault-tolerant quantum computing the Clifford gates can be implemented fault-tolerantly at low cost, often transversally, often transversally, whereas the $T$ gate has no transversal implementation.
To apply a $T$ gate fault-tolerantly, it is instead *injected* using a so-called ***Magic state***, via ***$T$-state injection***.
Since every ***$T$-state injection*** consumes one of this ***Magic state*** that has to be prepared, the amount of $T$ gates of a circuit is the dominant resource cost.

Any Clifford+$T$ circuit $U$ can be rearranged into a ***normal form*** that splits it into two parts,

$$U = \Big(\prod_{i=1}^{t} R(P_i)\Big)\, C,$$

a product of ***Exponents of Pauli gates*** $R(P_i)$, each containing a single $T$ gate, followed by an $n$-qubit Clifford $C$.


### Pauli-based computation (PBC) framework

**Pauli-based computation (PBC)** uses this ***Normal form***. Each Exponents of Pauli gates $R(P_i)$  is realised by ***Pauli product measurement*** and ***$T$-state injection*** using a ***Magic state***. 
That the quantum hardware only performs Pauli measurements, the Cliffords are handled classically. The cost is dominated ***Magic state*** that has to be prepared.

### Notebook structure

This notebook builds up the foundations of PBC step by step:

- **2** Background: Pauli operators, the Clifford group, and the $T$ gate
- **3** Exponents of Pauli gates
- **4** The normal form $U=\big(\prod_i R(P_i)\big)C$
- **5** The PBC framework


## 2. Background

### 2.1 Pauli Operators

<div style="border:1px solid #d0d7de; border-left:4px solid #2a6f97;
            border-radius:6px; background:#f6f9fc; padding:0.8em 1.2em; margin:1em 0;">

<strong style="color:#2a6f97;">Definition — Pauli operator</strong>

A ***Pauli operator*** $P$ over $n$ qubits is the tensor product of $n$ single-qubit Pauli matrices $P_i$ drawn from $\{I, X, Y, Z\}$:

$$P := P_1 \otimes P_2 \otimes \cdots \otimes P_n, \quad P_i \in \{I, X, Y, Z\}.$$

We write $\mathcal{P}_n$ for the set of all Pauli operators on $n$ qubits.

</div>

<div style="border:1px solid #d6cce6; border-left:4px solid #6a4c93;
            border-radius:6px; background:#f8f6fc; padding:0.8em 1.2em; margin:1em 0;">

<strong style="color:#6a4c93;">Property — Hermitian and squares to the identity</strong>

Every Pauli operator satisfies

$$P^\dagger = P \qquad\text{and}\qquad P^2 = I.$$

Its eigenvalues lie in $\{+1, -1\}$.

</div>

#### 2.1.1 Symplectic representation

Since $Y = iXZ$, we can represent $P$ with two bitvectors $x = (x_1 \ldots x_n)$ and $z = (z_1 \ldots z_n)$:

$$P(x, z) = i^{\langle x, z \rangle}\, X^x Z^z = P_1 \otimes P_2 \otimes \cdots \otimes P_n,$$

where the phase $i^{\langle x, z \rangle}$ (one factor of $i$ per $Y$) makes $P$ Hermitian, and each single-qubit factor $P_i$ is determined by:

| $x_i$ | $z_i$ | $P_i$ |
|--------|--------|--------|
| 0      | 0      | $I$    |
| 1      | 0      | $X$    |
| 1      | 1      | $Y$    |
| 0      | 1      | $Z$    |

For example, $P = X \otimes Y \otimes Z$ has $x = (1,1,0)$ and $z = (0,1,1)$, since:

$$\begin{align*}
P(x,z) &= i^{\langle (1,1,0),\,(0,1,1) \rangle} \, X^{(1,1,0)} Z^{(0,1,1)} \\
        &= i^1 \cdot (X \otimes X \otimes I)(I \otimes Z \otimes Z) \\
        &= X \otimes iXZ \otimes Z \\
        &= X \otimes Y \otimes Z
\end{align*}$$

> Instead of storing a Pauli operator over $n$ qubits as $n$ separate $2 \times 2$ matrices, in practice only the two length-$n$ bitvectors $x$ and $z$ are stored.


### 2.2 Pauli product measurements

<div style="border:1px solid #d0d7de; border-left:4px solid #6c757d;
            border-radius:6px; background:#f7f8f9; padding:0.8em 1.2em; margin:1em 0;">

<strong style="color:#6c757d;">Recall — projection postulate</strong>

A projective measurement is specified by a set of orthogonal projectors $\{\Pi_i\}$ with $\sum_i \Pi_i = I$ and $\Pi_i \Pi_j = \delta_{ij}\,\Pi_i$.

For a system in state $|\psi\rangle$, outcome $i$ occurs with probability $p_i = \langle\psi|\Pi_i|\psi\rangle$, and the state is updated to $\Pi_i|\psi\rangle/\sqrt{p_i}$.

</div>

<div style="border:1px solid #d0d7de; border-left:4px solid #2a6f97;
            border-radius:6px; background:#f6f9fc; padding:0.8em 1.2em; margin:1em 0;">

<strong style="color:#2a6f97;">Definition — Pauli product measurement</strong>

A ***Pauli product measurement*** measures a Pauli operator $M \in \mathcal{P}_n \setminus \{I\}$ as an observable. Since $M$ is Hermitian with $M^2 = I$, its eigenvalues are $\pm 1$ (§2.1), with orthogonal projectors

$$\Pi_\pm = \tfrac12\bigl(I \pm M\bigr), \qquad
\Pi_+ + \Pi_- = I,\quad \Pi_+\Pi_- = 0 .$$

Applied to a state $|\psi\rangle$, the measurement returns a **single bit** $m$ encoding the eigenvalue $(-1)^m$, and projects onto the corresponding eigenspace,

$$|\psi'\rangle = \frac{\Pi_{(-1)^m}\,|\psi\rangle}{\sqrt{p}}, \qquad
p = \langle\psi|\Pi_{(-1)^m}|\psi\rangle .$$

</div>

<div style="border:1px solid #f5d6a8; border-left:4px solid #e67e22;
            border-radius:6px; background:#fdf6ec; padding:0.8em 1.2em; margin:1em 0;">

<strong style="color:#e67e22;">❗ Important</strong>

This is a **non-demolition** measurement: for a Pauli product on $n$ qubits each $\Pi_\pm$ has rank $2^{\,n-1}$, i.e. it projects onto a whole eigenspace, not onto a single basis state. No individual qubit is read out.

</div>

### 2.2.1 Example: Pauli product measurment $Z\otimes Z$ on the state $|{+}{+}\rangle$

Now we look at a simple example of such a ***Pauli product measurement***: we measure
$Z\otimes Z$ on the state $|{+}{+}\rangle$.

<img src="assets/pauli_product_measurement_ex1.png" width="200">


<div style="border:1px solid #bcd9c5; border-left:4px solid #2d6a4f;
            border-radius:6px; background:#f4faf6; padding:0.8em 1.2em; margin:1em 0;">


$$Z\otimes Z=\begin{pmatrix}1&0&0&0\\0&-1&0&0\\0&0&-1&0\\0&0&0&1\end{pmatrix},$$

a diagonal matrix whose entries are the eigenvalues of the computational basis states $|00\rangle,|01\rangle,|10\rangle,|11\rangle$, namely $+1,-1,-1,+1$. So the $+1$ eigenspace is spanned by $\{|00\rangle,|11\rangle\}$ and the $-1$ eigenspace by $\{|01\rangle,|10\rangle\}$.

We measure the **product** state

$$|{+}{+}\rangle = \tfrac12\bigl(|00\rangle+|01\rangle+|10\rangle+|11\rangle\bigr).$$

**Explicit calculation.**

$$Z\otimes Z\,|{+}{+}\rangle = \tfrac12\bigl(|00\rangle-|01\rangle-|10\rangle+|11\rangle\bigr).$$

The $+1$-projector $\Pi_+ = \tfrac12\bigl(I + Z\otimes Z\bigr)$ then gives

$$\Pi_+|{+}{+}\rangle
= \tfrac12\Bigl(|{+}{+}\rangle + Z\otimes Z\,|{+}{+}\rangle\Bigr)
= \tfrac12\bigl(|00\rangle+|11\rangle\bigr),$$

with probability

$$p_+ = \langle{+}{+}|\Pi_+|{+}{+}\rangle = \bigl\|\tfrac12(|00\rangle+|11\rangle)\bigr\|^2 = \tfrac12 .$$

Normalising gives the post-measurement state, and the same computation with $\Pi_- = \tfrac12(I - Z\otimes Z)$ gives the other outcome:

$$m=0\ (+1):\ \tfrac{1}{\sqrt2}\bigl(|00\rangle+|11\rangle\bigr), \qquad
  m=1\ (-1):\ \tfrac{1}{\sqrt2}\bigl(|01\rangle+|10\rangle\bigr),$$

each with probability $\tfrac12$ — both **Bell states**.


> So we see that this ***Pauli product measurement*** has turned a product state into an *entangled* one.

</div>

The following code verifies this using `PauliProductMeasurement` from Qiskit: we prepare $|{+}{+}\rangle$, apply the Pauli product measurement $Z\otimes Z$, and sample the outcome $m$ (whose
eigenvalue is $(-1)^m$).
The two outcomes appear with probability $\approx\tfrac12$.


In [ ]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import PauliProductMeasurement
from qiskit.quantum_info import Pauli
from qiskit_aer import AerSimulator

qc = QuantumCircuit(2, 1)

# State |++> preparation
qc.h([0, 1])

# Pauli product measurement: Z⊗Z
qc.append(PauliProductMeasurement(Pauli("ZZ")), [0, 1], [0])

# sample the outcome m; the measured eigenvalue is (-1)^m
sim = AerSimulator(seed_simulator=42)
counts = sim.run(qc.decompose(), shots=4000).result().get_counts()

shots = sum(counts.values())
for m in sorted(counts):
    print(
        f"m={m}  (eigenvalue {(-1)**int(m):+d}):  {counts[m]:>4}  ({counts[m]/shots:.1%})"
    )

### 2.4 The Clifford Group

<div style="border:1px solid #d0d7de; border-left:4px solid #2a6f97;
            border-radius:6px; background:#f6f9fc; padding:0.8em 1.2em; margin:1em 0;">

<strong style="color:#2a6f97;">Definition — Clifford group</strong>

The ***Clifford group*** $\mathcal{C}_n$ over $n$ qubits is the set of unitaries that map every Pauli operator to a Pauli operator under conjugation:

$$\mathcal{C}_n := \{\, U \in U(2^n) \mid U P U^\dagger \in \mathcal{P}_n \;\;\forall P \in \mathcal{P}_n \,\},$$

where $\mathcal{P}_n$ is the Pauli group defined above.

</div>



#### 2.4.1 Generators

Up to a global phase, the $n$-qubit Clifford group is generated by Hadamard $H$, phase $S$, and CNOT gates applied to individual qubits or pairs of qubits:

$$\mathcal{C}_n = \langle H_i, S_i, \text{CNOT}_{i,j} \rangle,$$

where $H_i$ denotes $H$ acting on qubit $i$ and the identity on all other qubits (and analogously for $S_i$ and $\text{CNOT}_{i,j}$, where $i$ is the control qubit and $j$ is the target qubit).

In the computational basis we have:

$$H = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix}, \quad 
S = \begin{pmatrix} 1 & 0 \\ 0 & i \end{pmatrix}.$$

The Hadamard gate is self-inverse ($H^\dagger = H$), while the phase gate is not:

$$H^{\dagger} = H, \quad S^{\dagger} = \begin{pmatrix} 1 & 0 \\ 0 & -i \end{pmatrix}.$$

The CNOT gate acts on two qubits, designating one as the control and the other as the target. When the control is in state $|1\rangle$, the gate applies $X$ to the target, where $X$ is the bit-flip (NOT) operation that maps $|0\rangle \leftrightarrow |1\rangle$. In the basis $\{|00\rangle, |01\rangle, |10\rangle, |11\rangle\}$ with the first qubit as control:

$$\text{CNOT} = \begin{pmatrix} 1 & 0 & 0 & 0 \\ 0 & 1 & 0 & 0 \\ 0 & 0 & 0 & 1 \\ 0 & 0 & 1 & 0 \end{pmatrix}.$$

We write $\text{CNOT}_{i,j}$ for this gate placed with control on qubit $i$ and target on qubit $j$, acting as identity on all other qubits.


**Action on Pauli Operators**

By definition, a Clifford maps $\mathcal{P}_n \to \mathcal{P}_n$; the tables above illustrate illustrates how the conjugation of $H$, $S$ and $\text{CNOT}$ acts on the Pauli opertors

To see how Clifford gates (an element from the Clifford group) act on Pauli operators in conjugation, it suffices to compute how $H$ and $S$ conjugate the single-qubit Paulis $I$, $X$, $Y$, $Z$, since every Pauli operator is a tensor product of these.


<img src="assets/pauli_conjugation_hs.png" width="800">


For single-qubit gates, these rules follow from direct matrix multiplication (e.g., $H X H = Z$).

The action of $\text{CNOT}_{i,j}$ on single-Pauli inputs under conjugation $P \mapsto \text{CNOT}_{i,j}\, P\, \text{CNOT}_{i,j}^\dagger$ which is equal to $P \mapsto \text{CNOT}_{i,j}\, P\, \text{CNOT}_{i,j}$

<img src="assets/pauli_conjugation_cnot.png" width="900">



### 2.4.2 Properties

<div style="border:1px solid #d6cce6; border-left:4px solid #6a4c93;
            border-radius:6px; background:#f8f6fc; padding:0.8em 1.2em; margin:1em 0;">

<strong style="color:#6a4c93;">Fault tolerant benefit: applied trnsversally</strong>

The Clifford gates are inexpensive to protect against errors: in error correcting codes
such as the Steane code ([Steane, 1996](https://arxiv.org/abs/quant-ph/9601029)) they can
be applied *transversally* (acting on each physical qubit separately, which does not
spread errors).

</div>


<div style="border:1px solid #d6cce6; border-left:4px solid #6a4c93;
            border-radius:6px; background:#f8f6fc; padding:0.8em 1.2em; margin:1em 0;">

<strong style="color:#6a4c93;">Is not universal</strong>

But the Clifford group has only finitely many elements, so it can't approximate *every* unitary in $U(2^n)$. This ability fo the so called *universality* is what general quantum computation requires, and adding a single non-Clifford gate, the $T$ gate (next section), achieves it.

</div>

### 2.3 T gate

The non-Clifford gate that completes the Clifford gates to a universal set is the
***$T$ gate***.

<div style="border:1px solid #d0d7de; border-left:4px solid #2a6f97;
            border-radius:6px; background:#f6f9fc; padding:0.8em 1.2em; margin:1em 0;">

<strong style="color:#2a6f97;">Definition — T gate</strong>

The ***$T$ gate*** is the single-qubit phase gate

$$T = \begin{pmatrix} 1 & 0 \\ 0 & e^{i\pi/4} \end{pmatrix}, \qquad T^2 = S,$$

i.e. it applies a phase of $e^{i\pi/4}$ to $|1\rangle$ and leaves $|0\rangle$ unchanged.

</div>

Adding $T$ to the Clifford gates yields the **universal Clifford+T gate set**
$\{H, T, \text{CNOT}\}$: the Clifford gates alone are not universal (previous section),
but together with $T$ they generate a dense subset of $U(2^n)$, so any unitary can be
approximated to arbitrary precision.


<div style="border:1px solid #d6cce6; border-left:4px solid #6a4c93;
            border-radius:6px; background:#f8f6fc; padding:0.8em 1.2em; margin:1em 0;">

<strong style="color:#6a4c93;">Property — T escapes the Pauli group</strong>

Conjugating a Pauli by a Clifford gate gives another Pauli. Conjugating by $T$ does
**not**:

$$T X T^\dagger = \tfrac{1}{\sqrt{2}}(X + Y) \notin \mathcal{P}_1 .$$

So Clifford gates keep the description *inside* the Pauli group $\mathcal{P}_n$, where
it stays efficiently trackable, while each $T$ gate maps it *outside*, which is used to construct a universal gate set.

</div>

#### Why T gate is "expensive"

<div style="border:1px solid #f5d6a8; border-left:4px solid #e67e22;
            border-radius:6px; background:#fdf6ec; padding:0.8em 1.2em; margin:1em 0;">

<strong style="color:#e67e22;">❗ Important</strong>

Unlike the Clifford gates, $T$ cannot be implemented *transversally* in a fault-tolerant code, so it cannot be applied directly to logical qubits. Instead it is realised by **$T$-state injection**, which we will see later, which makes it computational costly.

</div>


## 3 Exponents of Pauli Gates


Now we look deeper of the form of the T gate

$T$ gate can be expressed as an exponent. We can write the $T$ gate as following

$T = e^{\,i\frac{\pi}{8}(I - Z)} = \tfrac{1}{2}\big(1 + e^{i\pi/4}\big)I + \tfrac{1}{2}\big(1 - e^{i\pi/4}\big)Z,$

This is a $\tfrac{\pi}{4}$ rotation about the $Z$ axis.

<img src="assets/bloch_t_rotation.png" width="400">


Replacing $Z$ by another single-qubit Pauli gives the same gate seen along a different axis, each a $\tfrac{\pi}{4}$ rotation up to a global phase:

$R(X) = e^{\,i\frac{\pi}{8}(I - X)} = H\,{\color{red}T}\,H \qquad (\tfrac{\pi}{4}\text{ rotation about the } X \text{ axis}),$

$R(Y) = e^{\,i\frac{\pi}{8}(I - Y)} = S\,H\,{\color{red}T}\,H\,S^\dagger \qquad (\tfrac{\pi}{4}\text{ rotation about the } Y \text{ axis}),$

$R(Z) = e^{\,i\frac{\pi}{8}(I - Z)} = {\color{red}T} \qquad (\tfrac{\pi}{4}\text{ rotation about the } Z \text{ axis}).$

We see that each can be expressed with only one ${\color{red}T}$.



We looked at the rotation on one qubit if we extend on n- qubit space we become the so called ***exponent of a Pauli gate***

<div style="border:1px solid #d0d7de; border-left:4px solid #2a6f97;
            border-radius:6px; background:#f6f9fc; padding:0.8em 1.2em; margin:1em 0;">

<strong style="color:#2a6f97;">Definition — Exponent of a Pauli gate</strong>

For $P \in \mathcal{P}_n \setminus \{I_{2^n}\}$, the ***exponent of a Pauli gate*** about
$\pm P$ is

$$R(\pm P) := e^{\,i\frac{\pi}{8}(I \mp P)}
   = \tfrac{1}{2}\big(1 + e^{i\pi/4}\big)I_{2^n} \pm \tfrac{1}{2}\big(1 - e^{i\pi/4}\big)P .$$

These are $\tfrac{\pi}{4}$ rotations around $P$, up to a global phase.

</div>


<div style="border:1px solid #e6d77a; border-left:4px solid #c9a227;
            border-radius:6px; background:#fdfae6; padding:0.8em 1.2em; margin:1em 0;">

<strong style="color:#a07c00;">Key fact — T-count one</strong>

Every exponent of a Pauli gate can be implemented with Clifford gates and a **single $T$**, so it has **$T$-count one**.

</div>



**Cliffords map rotations to rotations.** For any Clifford $C \in \mathcal{C}_n$,

$$C\, R(P)\, C^\dagger = R\big(C P C^\dagger\big).$$

Conjugating a exponent of a Pauli gate by a Clifford gives another exponent of a Pauli gate, with axis the conjugated Pauli $C P C^\dagger$. Because conjugation stays inside the Pauli operators, the right-hand side is again a valid rotation. 

<div style="border:1px solid #d6cce6; border-left:4px solid #6a4c93;
            border-radius:6px; background:#f8f6fc; padding:0.8em 1.2em; margin:1em 0;">

<strong style="color:#6a4c93;">Consequence — push Cliffords to the right</strong>

This identity implies

$$\textcolor{green}{\mathbf{C}}\, R(P) = R\big(C P C^\dagger\big)\, \textcolor{green}{\mathbf{C}} = R(P')\, \textcolor{green}{\mathbf{C}}, \qquad \text{where } P' = C P C^\dagger \in \mathcal{P}_n.$$

which means we can move a Clifford $\textcolor{green}{\mathbf{C}}$ **to the right** past a rotation by conjugating its
Pauli. Repeating this collects every Clifford on the right — the basis of the **normal
form** in the next section.

</div>


## 4 Normal Form for Clifford+T Circuits

### 4.1 Statement and Meaning
<div style="border:1px solid #cfd3ec; border-left:4px solid #3b3f8c;
            border-radius:6px; background:#f5f6fc; padding:0.8em 1.2em; margin:1em 0;">

<strong style="color:#3b3f8c;">Theorem — Normal form</strong>

Let $U$ be a $2^n \times 2^n$ unitary that can be represented by a Clifford+$T$ circuit
($\{H, S, \mathrm{CNOT}, T\}$). Then there exist $P_1, P_2, \ldots, P_k \in \mathcal{P}_n$
and $C \in \mathcal{C}_n$ such that

$$U = \left(\prod_{i=1}^{k} R(P_i)\right) \cdot C.$$

This normal form partitions the circuit into two parts:

1. the first part, $\prod_{i=1}^{k} R(P_i)$, consists only of exponents of Pauli gates;
2. the second part, $C$, is a single Clifford operation.

</div>


<div style="border:1px solid #e6d77a; border-left:4px solid #c9a227;
            border-radius:6px; background:#fdfae6; padding:0.8em 1.2em; margin:1em 0;">

<strong style="color:#a07c00;">Key fact — T-count: k</strong>

Since exponent of a Pauli gate can be implemented with Clifford gates and a single $T$, so it has **$T$-count: k**.

</div>


### 4.2 Proof

Suppose $U$ is given as a Clifford+$T$ circuit with exactly $k$ $T$ gates. The Clifford
gates between (and around) the $T$ gates group into Clifford operators
$C_1, C_2, \ldots, C_{k+1}$ (some possibly the identity), so the circuit reads

$$U = C_1\, R(P_1)\, C_2\, R(P_2)\cdots C_k\, R(P_k)\, C_{k+1},$$

where each $R(P_i)$ is the $i$-th $T$ gate written as an exponent of a Pauli gate. A $T$
gate is $T = R(Z)$ acting on a single qubit $j_i$, so each $P_i = \pm Z_{j_i}$ is a
single-qubit Pauli, hence $P_i \in \mathcal{P}_n$.

Now use the identity *Cliffords map rotations to rotations* (§3) in the form

$$C\, R(P) = R\big(C P C^\dagger\big)\, C,$$

which follows from $C\,R(P)\,C^\dagger = R(CPC^\dagger)$. It lets us **move a Clifford to
the right past a rotation**, at the cost of conjugating the Pauli, $P \mapsto CPC^\dagger
\in \mathcal{P}_n$.

Apply this repeatedly, pushing every $C_i$ rightward through all the rotations to its
right. Each $R(P_i)$ is replaced by $R(P_i')$ with $P_i' \in \mathcal{P}_n$ (a conjugate
of $P_i$), and all the Clifford factors collect into a single Clifford at the end,

$$C = C_1 C_2 \cdots C_{k+1} \in \mathcal{C}_n.$$

This yields

$$U = \left(\prod_{i=1}^{k} R(P_i')\right) C,$$



### 4.3 Algorithm (applied example)




In [ ]:
from viz import normal_form_steps_to_latex, draw_circuit
from IPython.display import display

u = [
    ("T", [0]),
    ("H", [0]),
    ("T", [0]),
    ("H", [0]),
    ("CNOT", [0, 1]),
    ("T", [1]),
    ("H", [2]),
    ("S", [2]),
]
display(draw_circuit(u, 3))
display(normal_form_steps_to_latex(u, 3))

## 5. Framework: Pauli based Computation

From the normal form theorem we know that any Clifford+T unitary factors as a product of rotations followed by a single Clifford, $U = \big(\prod_i R(P_i)\big)\, C$. The Clifford part $C$ is not the difficulty: Clifford gates have transversal, and therefore fault-tolerant, implementations on the standard quantum error-correcting codes. The rotations are the obstacle. Each $R(P_i)$ contains exactly one $T$ gate, and unlike the Cliffords the $T$ gate has no transversal implementation on these codes, so it cannot be applied as directly.
<img src="assets/t_gate_not_direct.png" width="200">


So we look at first in a isolated way how to apply a T. For this we first turn to ***magic states***

### 5.1  Magic state

A ***magic state*** is the fixed single-qubit state

$$|T\rangle = T|+\rangle = \frac{1}{\sqrt{2}}\left(|0\rangle + e^{i\pi/4}|1\rangle\right).$$

Magic states are made in special parts of the hardware called factories. A factory uses only Clifford operations to turn many noisy copies into a few very accurate ones, so magic states can be made as precise as needed without ever applying a $T$ gate. This is why we may treat them as ideal in this notebook. But creating them 

**Reference.** S. Bravyi and A. Kitaev, *Universal quantum computation with ideal Clifford gates and noisy ancillas*, Phys. Rev. A **71**, 022316 (2005), [arXiv:quant-ph/0403025](https://arxiv.org/abs/quant-ph/0403025).

### T-state injection (applying a T gate without using a T gate)

The circuit below shows how to produce $T|\psi\rangle$ without applying $T$ directly.


<img src="assets/t_state_injection.png" width="700">

**Why not apply the $T$ gate directly?** In a fault-tolerant quantum computer the logical qubits are not single physical qubits. They are encoded into many physical qubits by an error-correcting code, and every operation must be carried out so that a single physical error cannot grow into a logical error. The safe way to do this is a ***transversal*** gate, one that acts on each physical qubit of the code block on its own and therefore never couples errors between them.

The Clifford gates can be made fault-tolerant on the usual codes. The $T$ gate is the problem. Since the set of Clifford gates together with $T$ is universal, the ***Eastin-Knill theorem*** guarantees that not all of these gates can be transversal at the same time, and on the standard codes it is exactly the $T$ gate that fails. So $T$ cannot be applied directly on encoded data as a simple fault-tolerant operation.

The way around this is to move the non-Clifford work off the data and into a resource state. We prepare the magic state $|T\rangle$ separately and then apply the $T$ rotation by ***injection***, using only a CNOT, a measurement, and a Clifford correction, all of which are fault-tolerant. The data qubit never sees a physical $T$ gate. This is not circular: the magic state is prepared once, offline, as a standard resource and cleaned up by the factory, so the expensive step is paid in the factory and not during the computation.

**The calculation.** Let the data qubit be $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$ and the magic qubit be $|T\rangle = \tfrac{1}{\sqrt{2}}(|0\rangle + e^{i\pi/4}|1\rangle)$. A CNOT with the data as control and the magic qubit as target gives

$$\mathrm{CNOT}\,\big(|\psi\rangle \otimes |T\rangle\big) = \tfrac{1}{\sqrt{2}}\Big( \alpha|0\rangle\,(|0\rangle + e^{i\pi/4}|1\rangle) + \beta|1\rangle\,(|1\rangle + e^{i\pi/4}|0\rangle) \Big).$$

Sorting the terms by the state of the magic qubit,

$$= \tfrac{1}{\sqrt{2}}\Big( \big(\alpha|0\rangle + \beta e^{i\pi/4}|1\rangle\big)\,|0\rangle \;+\; \big(\alpha e^{i\pi/4}|0\rangle + \beta|1\rangle\big)\,|1\rangle \Big).$$

Now measure the magic qubit.

If the outcome is $0$, the data collapses to

$$\alpha|0\rangle + \beta e^{i\pi/4}|1\rangle = T|\psi\rangle,$$

which is exactly the desired result, with no correction needed.

If the outcome is $1$, the data is $\alpha e^{i\pi/4}|0\rangle + \beta|1\rangle$. Applying the Clifford correction $S = \mathrm{diag}(1, i)$ gives

$$S\big(\alpha e^{i\pi/4}|0\rangle + \beta|1\rangle\big) = \alpha e^{i\pi/4}|0\rangle + \beta\, i\,|1\rangle = e^{i\pi/4}\big(\alpha|0\rangle + \beta e^{i\pi/4}|1\rangle\big) = e^{i\pi/4}\,T|\psi\rangle,$$

which is $T|\psi\rangle$ up to an irrelevant global phase.

In both cases the data ends in the state $T|\psi\rangle$, and the only operations used were a CNOT, a measurement, and a conditional $S$. All of them are Clifford, so the $T$ rotation was applied without ever using a $T$ gate.

### 5.2 Generalisation to $R(P)$ 

We now return to the original problem of applying each rotation $R(P_i)$ in a fault-tolerant way. We know that every $R(P)$, no matter how many of the $n$ qubits $P$ acts on, contains exactly one $T$ gate. Its non-Clifford cost is therefore always the same, a single $T$, so we can treat all rotations with one general construction that consumes a single magic state $|T\rangle$.

The difficulty is that $R(P)$ itself is not a Clifford operator,

$R(P) = \exp\!\left[\,i\frac{\pi}{8}\,(I_{n} - P)\right] \in \mathcal{C}_n.$

since it is a $\pi/8$ rotation about $P$ and contains one T gate.

It's square, however, is a Clifford: squaring doubles the angle from $\pi/8$ to $\pi/4$, and a $\pi/4$ rotation about a Pauli lies in $\mathcal{C}_n$,

$S(P) := R(P)^2 = \exp\!\left[\,i\frac{\pi}{4}\,(I_{n} - P)\right] \in \mathcal{C}_n.$

We use now this and the T state injection to create a $R(P)|\psi\rangle$

<img src="assets/pbc_framework.png" width="900">

**Initial state: $|\Psi_0\rangle$**


**joint projective measurement: $|\Psi_1\rangle$**
We apply a **joint projective measurement** of the Pauli observable
$M = P \otimes Z$ to the input state $|\Psi_0\rangle = |\psi\rangle \otimes |T\rangle$,
where $|\psi\rangle$ is the $n$-qubit data state, $P$ is an $n$-qubit Pauli operator
($P = P^\dagger$, $P^2 = I$), and $|T\rangle = \tfrac{1}{\sqrt 2}\bigl(|0\rangle + e^{i\pi/4}|1\rangle\bigr)$.

exact formula: 

Because $P^2 = I$ and $Z^2 = I$, the observable satisfies $M = M^\dagger$ and
$M^2 = I$, so all it's eigenvalues are in $\{+1,-1\}$

$$M = (+1)\,\Pi_+ + (-1)\,\Pi_-, \qquad
\Pi_\pm = \tfrac{1}{2}\bigl(I \pm P\otimes Z\bigr),$$

where $\Pi_\pm^2 = \Pi_\pm$, $\Pi_+\Pi_- = 0$, $\Pi_+ + \Pi_- = I$, and each
$\Pi_\pm$ has rank $2^{\,n}$ (half the Hilbert space).

By the projection postulate the measurement yields a single outcome
$s \in \{+1,-1\}$, recorded as the bit $m_1$ with $s = (-1)^{m_1}$, with probability

$$p_s = \langle \Psi_0 |\, \Pi_s \,|\Psi_0\rangle
      = \tfrac{1}{2}\Bigl(1 + s\,\langle\psi|P|\psi\rangle\,\langle T|Z|T\rangle\Bigr)
      = \tfrac{1}{2},$$

since $\langle T|Z|T\rangle = 0$; the outcome is therefore unbiased for *every*
input $|\psi\rangle$. The post-measurement state is

$$|\Psi_1\rangle = \frac{\Pi_s\,|\Psi_0\rangle}{\sqrt{p_s}}.$$



**Measuring $|T\rangle$ and reset: $|\Psi_2\rangle$**

To make the states explicit, split the data state into the $\pm1$ eigenspaces of
$P$,
$$|\psi\rangle = |\psi_+\rangle + |\psi_-\rangle, \qquad
  |\psi_\pm\rangle = \tfrac12(I\pm P)\,|\psi\rangle, \quad P|\psi_\pm\rangle = \pm|\psi_\pm\rangle.$$
In this basis the post-measurement state of the previous step reads
$$m_1=0:\ |\Psi_1\rangle = |\psi_+\rangle|0\rangle + e^{i\pi/4}|\psi_-\rangle|1\rangle,
\qquad
  m_1=1:\ |\Psi_1\rangle = e^{i\pi/4}|\psi_+\rangle|1\rangle + |\psi_-\rangle|0\rangle.$$

We read out the magic state in the $X$ basis — apply $H$ and measure in $Z$,
recording the bit $m_2$ — and then reset it to $|0\rangle$. Writing the magic
register in the $X$ basis and projecting onto outcome $m_2$ leaves
$$m_1=0:\ |\Psi_2\rangle = \bigl(|\psi_+\rangle + (-1)^{m_2}e^{i\pi/4}|\psi_-\rangle\bigr)\otimes|0\rangle,$$
$$m_1=1:\ |\Psi_2\rangle = \bigl((-1)^{m_2}e^{i\pi/4}|\psi_+\rangle + |\psi_-\rangle\bigr)\otimes|0\rangle,$$
up to an irrelevant global phase. The target of the gadget is the $\pi/8$ rotation
$$R(P)\,|\psi\rangle = |\psi_+\rangle + e^{i\pi/4}|\psi_-\rangle,$$
so $|\Psi_2\rangle$ already equals $R(P)|\psi\rangle\otimes|0\rangle$ for
$m_1=m_2=0$, and otherwise differs from it by a Pauli/Clifford byproduct that the
two outcome-controlled corrections remove.

**Optional correction with $S(P)$: $|\Psi_3\rangle$**

The bit $m_1$ controls a correction by the Clifford $S(P)=e^{\,i\frac{\pi}{4}(I-P)}$,
which acts as $|\psi_+\rangle\mapsto|\psi_+\rangle$, $|\psi_-\rangle\mapsto i\,|\psi_-\rangle$.

*Case $m_1=1$.* In this branch the weight $e^{i\pi/4}$ sits on the wrong
component, so we apply $S(P)$ to the data register:
$$|\Psi_3\rangle = \bigl(S(P)\otimes I\bigr)\,|\Psi_2\rangle
= \bigl((-1)^{m_2}|\psi_+\rangle + e^{i\pi/4}|\psi_-\rangle\bigr)\otimes|0\rangle$$
(up to a global phase).

*Case $m_1=0$.* No correction is applied, $|\Psi_3\rangle = |\Psi_2\rangle$.

In both cases the data register is now $R(P)|\psi\rangle$ up to the single
remaining byproduct $P^{m_2}$:
$$|\Psi_3\rangle = \bigl(P^{m_2} R(P)\,|\psi\rangle\bigr)\otimes|0\rangle
\quad(\text{up to a global phase}).$$

**Optional correction with $P$: $|\Psi_4\rangle$**

The bit $m_2$ controls a correction by the Pauli $P$, which flips the sign of the
$|\psi_-\rangle$ component, $P|\psi_\pm\rangle = \pm|\psi_\pm\rangle$.

*Case $m_2=1$.* We apply $P$, removing the last byproduct:
$$|\Psi_4\rangle = (P\otimes I)\,|\Psi_3\rangle
= R(P)\,|\psi\rangle \otimes |0\rangle \quad(\text{up to a global phase}).$$

*Case $m_2=0$.* No correction is applied, and already
$|\Psi_4\rangle = |\Psi_3\rangle = R(P)\,|\psi\rangle\otimes|0\rangle$.

In every one of the four measurement branches the gadget therefore outputs
$$|\Psi_4\rangle = R(P)\,|\psi\rangle \otimes |0\rangle,$$
i.e. the $\pi/8$ rotation $R(P)$ applied to the data, with the magic register
returned to $|0\rangle$.







### 5.3 The Full Framework

The circuit above is a single step: one magic state, one joint Pauli
measurement, one non-Clifford operation $R(P)$ on the data. The full
computation $U = \big(\prod_{i=1}^{t} R(P_i)\big)\,C$ is just this step repeated
$t$ times on a register of $t$ magic states. The Clifford $C$ is commuted to the
end, where it only decides which Pauli $P_i$ each step measures,
$P_i \mapsto C P_i C^{\dagger}$, all tracked classically. So the whole program
is simply a sequence of joint Pauli measurements on magic states.

# References

The notebook is based on the following lecture notes:

1. *Fault-Tolerant Quantum Computing* (CS-724), lecture notes, EPFL, 2026.
2. M. Soeken, *Advanced Logic Synthesis and Quantum Computation*, typed lecture notes, version 1.414.